# Spark SQL: Temporary Views, Global Views, and SQL Execution

**Module**: 03_dataframe_and_spark_sql  
**Notebook**: 04_spark_sql.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Writing declarative ANSI SQL queries against Spark DataFrames, managing temporary and global views, and using Common Table Expressions (CTEs).
- **Why does this topic exist?**: SQL is the universal language of data analytics. Spark SQL compiles ANSI SQL into the exact same optimized Catalyst physical plans as the DataFrame API.
- **What problem does it solve?**: Knowing when to use raw SQL versus programmatic DataFrame transformations.
- **Where is it used in real-world Data Engineering?**: Migrating legacy SQL warehouse queries (Snowflake, BigQuery, Oracle) to Apache Spark.

---

## 2. Spark SQL vs DataFrame API Equivalence

```
User ANSI SQL Query:
"SELECT region, sum(revenue) FROM sales GROUP BY region"
                      |
                      v (Parsed by Catalyst SQL Parser)
          Parsed Logical Plan
                      ^
                      | (Built by DataFrame API)
User DataFrame API Call:
df.groupBy("region").agg(sum("revenue"))
```
> Both APIs compile into the **exact same Physical Plan and execute identically**!

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Spark_SQL_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Registering Views and Running SQL Queries

In [ ]:
# Register retail datasets as SQL views
orders = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/orders.csv")
customers = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/customers.csv")

orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")

# Query with CTE and aggregations
sql_query = """
WITH customer_revenue AS (
    SELECT
        o.customer_id,
        c.region,
        c.segment,
        SUM(o.quantity * o.unit_price) AS total_spent,
        COUNT(o.order_id) AS total_orders
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.status = 'Completed'
    GROUP BY o.customer_id, c.region, c.segment
)
SELECT
    region,
    segment,
    COUNT(customer_id) AS active_customers,
    ROUND(SUM(total_spent), 2) AS regional_revenue,
    ROUND(AVG(total_spent), 2) AS avg_customer_spend
FROM customer_revenue
GROUP BY region, segment
ORDER BY regional_revenue DESC
"""

result_df = spark.sql(sql_query)
result_df.show(10)

---

## Experiment: Verifying Physical Plan Parity: SQL vs DataFrame API

- **1. What are we testing?**: Prove that an identical SQL query and DataFrame call generate the exact same Catalyst physical plan.
- **2. What do we expect?**: Both explain() outputs will show identical physical operators.

In [ ]:
# SQL approach
plan_sql = spark.sql("SELECT status, count(*) as cnt FROM orders GROUP BY status")

# DataFrame approach
plan_df = orders.groupBy("status").count()

print("=== SQL Physical Plan ===")
plan_sql.explain()

print("\n=== DataFrame Physical Plan ===")
plan_df.explain()

**3. What happened?**: Both approaches generated identical HashAggregate and Exchange physical operators.  
**4. Why did it happen?**: Spark SQL parses SQL into the same Catalyst AST (Abstract Syntax Tree) that DataFrame methods build.  
**5. Practical Engineering Lesson**: Choose SQL or DataFrame API based on readability and team preference; performance is identical.

---

## Practical Exercises

### Exercise 1 (Beginner): Monthly Revenue Trend Query
**Problem Statement**: Write an ANSI SQL query to compute monthly completed revenue and order count for 2025.  
**Input / Context**: orders view

In [ ]:
# Exercise 1: Monthly revenue in SQL

### Exercise 2 (Intermediate): Top 5 Products per Channel Query
**Problem Statement**: Write a SQL query using window function DENSE_RANK() to find the top 3 selling products per channel.  
**Input / Context**: orders view

In [ ]:
# Exercise 2: Top products per channel

### Exercise 3 (Advanced): Multi-Table SQL Join with Customer Loyalty
**Problem Statement**: Query revenue by loyalty_tier and channel, showing percentage share of total revenue.  
**Input / Context**: orders and customers views

In [ ]:
# Exercise 3: Revenue share by loyalty

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Monthly Revenue Trend Query):
```python
spark.sql("""
    SELECT 
        date_format(order_ts,'yyyy-MM') AS order_month,
        COUNT(order_id) AS total_orders,
        ROUND(SUM(quantity * unit_price), 2) AS monthly_revenue
    FROM orders
    WHERE status ='Completed'
    GROUP BY date_format(order_ts,'yyyy-MM')
    ORDER BY order_month
""").show()
```

#### Solution 2 (Top 5 Products per Channel Query):
```python
spark.sql("""
    WITH ranked_products AS (
        SELECT 
            channel,
            product_id,
            SUM(quantity) AS total_units,
            DENSE_RANK() OVER (PARTITION BY channel ORDER BY SUM(quantity) DESC) as rnk
        FROM orders
        GROUP BY channel, product_id
)
    SELECT * FROM ranked_products WHERE rnk <= 3 ORDER BY channel, rnk
""").show()
```

#### Solution 3 (Multi-Table SQL Join with Customer Loyalty):
```python
spark.sql("""
    SELECT 
        c.loyalty_tier,
        o.channel,
        ROUND(SUM(o.quantity * o.unit_price), 2) AS tier_revenue
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.loyalty_tier, o.channel
    ORDER BY tier_revenue DESC
""").show(10)
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **SQL vs DataFrame**: Identical execution plans and performance under Catalyst.
- **Temp Views**: Session-scoped virtual tables registered in internal Catalog.
- **Advanced SQL**: Supports full ANSI SQL including CTEs, Window functions, and nested queries.

### Essential Technical Terms
- `createOrReplaceTempView`: Registers a DataFrame as a temporary SQL view.
- `spark.sql()`: Executes an SQL query string and returns a DataFrame.
- `CTE`: Common Table Expression (WITH clause).

### Common Engineering Mistakes
- **Assuming SQL is slower than DataFrames**: False: both compile to the exact same bytecode.
- **SQL Injection in Unsanitized Strings**: Avoid dynamic f-string queries with untrusted user input.

### Interview Questions on This Topic
- **Q: Is Spark SQL slower than the DataFrame API?**
  - *A*: No, both produce identical Catalyst logical plans and compile into the same Tungsten physical bytecode.
- **Q: How long do temporary views last?**
  - *A*: Only for the lifetime of that specific SparkSession.

### What to Study Next
Open 03_dataframe_and_spark_sql/05_joins_and_aggregations.ipynb to master distributed joins.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')